# U1-06 · Tensores y primera red en TensorFlow

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartados 1.9 y 1.10*

En Colab, activa la GPU si quieres ir más rápido: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Sin ella funciona igual, algo más despacio.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges, con licencia Creative Commons Reconocimiento-CompartirIgual 3.0.

## 1. Tensores y derivación automática

In [ ]:
import numpy as np
import tensorflow as tf

x = tf.constant([[1., 2.], [3., 4.]])
print(x.shape, x.dtype, "| suma:", tf.reduce_sum(x).numpy())
print("desde NumPy y de vuelta:",
      tf.convert_to_tensor(np.eye(2)).numpy().tolist())

w = tf.Variable(3.0)                     # un parámetro
with tf.GradientTape() as cinta:         # graba las operaciones
    perdida = (w * 2.0 - 1.0) ** 2       # (2w - 1)^2
print("derivada automática:", cinta.gradient(perdida, w).numpy(),
      "| a mano: 4(2w-1) =", 4 * (2 * 3.0 - 1))

W = tf.Variable(tf.random.normal((784, 10), seed=0))
lote = tf.random.uniform((32, 784), seed=1)
with tf.GradientTape() as cinta:
    perdida = tf.reduce_mean(tf.square(lote @ W))
print("gradiente respecto a W:", cinta.gradient(perdida, W).shape)

## 2. La red en scikit-learn

In [ ]:
import time
import keras
from sklearn.neural_network import MLPClassifier

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X, Xp = x_ent.reshape(60000, -1) / 255, x_pru.reshape(10000, -1) / 255
t = time.perf_counter()
red = MLPClassifier(hidden_layer_sizes=(128,), max_iter=5,
                    batch_size=32, random_state=0).fit(X, y_ent)
print(f"scikit-learn 784-128-10: acierto {red.score(Xp, y_pru):.4f}"
      f" ({time.perf_counter() - t:.0f} s, 5 épocas)")
print("parámetros:", sum(c.size for c in red.coefs_)
      + sum(b.size for b in red.intercepts_))

## 3. La primera red con TensorFlow

In [ ]:
import time
import numpy as np
import keras

keras.utils.set_random_seed(0)              # resultados repetibles
(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
x_ent, x_pru = x_ent / 255.0, x_pru / 255.0  # píxeles entre 0 y 1

def red(ocultas):
    capas = [keras.Input(shape=(28, 28)), keras.layers.Flatten()]
    capas += [keras.layers.Dense(n, activation="relu")
              for n in ocultas]
    capas += [keras.layers.Dense(10, activation="softmax")]
    modelo = keras.Sequential(capas)
    modelo.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
    return modelo

for ocultas in ([], [128]):           # sin capa oculta y con una
    modelo = red(ocultas)
    t0 = time.perf_counter()
    h = modelo.fit(x_ent, y_ent, epochs=5, batch_size=32,
                   validation_split=0.1, verbose=0)
    seg = time.perf_counter() - t0
    perdida, acierto = modelo.evaluate(x_pru, y_pru, verbose=0)
    print(f"capas ocultas {ocultas}: {modelo.count_params():,} "
          f"parámetros, {seg:.0f} s, validación "
          f"{h.history['val_accuracy'][-1]:.4f}, "
          f"prueba {acierto:.4f}")

prob = modelo.predict(x_pru, verbose=0)
pred = prob.argmax(axis=1)
fallos = np.flatnonzero(pred != y_pru)
print(len(fallos), "fallos de 10 000; por época, validación:",
      np.round(h.history["val_accuracy"], 4))

import matplotlib.pyplot as plt
epocas = range(1, 6)
plt.plot(epocas, h.history["accuracy"], "o-", label="entrenamiento")
plt.plot(epocas, h.history["val_accuracy"], "o-", label="validación")
plt.xlabel("época"); plt.ylabel("acierto"); plt.xticks(epocas)
plt.legend()
plt.show()  # plt.savefig("curvas.pdf")
print("entrenamiento por época:", np.round(h.history["accuracy"], 4))

fig, ejes = plt.subplots(1, 8, figsize=(4.4, 0.9))
for eje, i in zip(ejes, fallos[:8]):
    eje.imshow(x_pru[i])
    eje.set_title(f"{y_pru[i]} → {pred[i]}\n{prob[i].max():.0%}",
                  fontsize=6, loc="center", fontweight="normal")
    eje.axis("off")
plt.show()  # fig.savefig("fallos.pdf")
pares = np.bincount(y_pru[fallos] * 10 + pred[fallos], minlength=100)
for c in np.argsort(pares)[::-1][:3]:
    print(f"real {c // 10} tomado por {c % 10}: {pares[c]} veces")

**Prueba.** Cambia la capa oculta a 32 y a 512 neuronas, y las épocas a 10. Anota parámetros, tiempo y acierto en prueba.

## 4. Guardar, cargar y usar la red

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import keras

keras.utils.set_random_seed(0)
(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
x_ent, x_pru = x_ent / 255.0, x_pru / 255.0
modelo = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")])
modelo.compile(optimizer="adam", metrics=["accuracy"],
               loss="sparse_categorical_crossentropy")
modelo.fit(x_ent, y_ent, epochs=5, validation_split=0.1, verbose=0)

modelo.save("mnist_densa.keras")               # guardar
copia = keras.models.load_model("mnist_densa.keras")  # y cargar
p = copia.predict(x_pru, verbose=0)
print(f"fichero: {os.path.getsize('mnist_densa.keras') / 1e6:.1f}"
      f" MB; mismas predicciones: "
      f"{np.allclose(p, modelo.predict(x_pru, verbose=0))}")
print("una imagen:", y_pru[0], "->", p[0].argmax(),
      f"con probabilidad {p[0].max():.4f}")

pred, segura = p.argmax(axis=1), p.max(axis=1) >= 0.99
print(f"acierto global {(pred == y_pru).mean():.4f}")
print(f"seguras (≥ 99 %): {segura.mean():.1%} de las imágenes, "
      f"acierto {(pred == y_pru)[segura].mean():.4f}")
print(f"resto: acierto {(pred == y_pru)[~segura].mean():.4f}")

M = np.zeros((10, 10), int)
np.add.at(M, (y_pru, pred), 1)                 # matriz de confusión
plt.figure(figsize=(2.6, 2.4))
plt.imshow(np.log1p(M)); plt.xticks(range(10)); plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        if M[i, j]:
            plt.text(j, i, M[i, j], ha="center", va="center",
                     fontsize=4.5, color="w" if i == j else "k")
plt.xlabel("predicho"); plt.ylabel("real")
plt.show()  # plt.savefig("confusion.pdf")
print("aciertos por dígito:", (np.diag(M) / M.sum(1)).round(3))

**Ejercicio.** Cambia el umbral de seguridad de 0,99 a 0,9 y a 0,999. ¿Qué parte de las imágenes quedaría para revisión humana?

> Deng, L. (2012). The MNIST database of handwritten digit images for machine learning research. *IEEE Signal Processing Magazine*, 29(6), 141–142. https://doi.org/10.1109/MSP.2012.2211477
>
> Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning. ISBN 978-1-61729-686-4